# POST SSE 이벤트 설계

POST 본문으로 질문을 보내고 네이티브 SSE의 `event`, `id`, `data` 필드를 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

### 이 준비 셀이 하는 일

이 준비 셀은 `Path.cwd().resolve()`로 현재 폴더를 절대 경로로 읽고, 현재 폴더와 부모 폴더에서 `agent_service`가 있는 첫 위치를 `next(..., None)`로 찾습니다. 찾지 못하면 `assert`가 실행을 중단합니다. 찾은 뒤 `os.chdir(lab)`이 이후 import와 파일 접근의 기준 폴더를 바꾸고, 마지막 출력은 실제 선택된 폴더 이름을 확인합니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 기본 실행

TestClient가 실제 ASGI 스트림을 끝까지 읽어 전송 형식을 보여 줍니다.

### 바로 다음 코드를 읽는 방법

SSE(Server-Sent Events)는 한 HTTP 응답 안에서 `event`, `id`, `data` 줄을 묶어 사건을 순서대로 보내는 텍스트 형식입니다. 이 실습은 POST 본문으로 질문을 받은 뒤 `token` 사건을 여러 번 보내고 마지막에 `done` 사건을 한 번 보냅니다. 빈 줄은 사건과 사건의 경계입니다. 정규식으로 응답을 나눈 다음 첫 사건의 종류와 식별자, 마지막 사건의 종류를 각각 검사합니다.

In [2]:
from fastapi.testclient import TestClient
from agent_service.app import app
client = TestClient(app)
headers = {'Authorization': 'Bearer local-demo-token'}
response = client.post('/v1/chat/stream', headers=headers,
    json={'question': 'SSE 확인', 'thread_id': 'sse-1'})
print(response.status_code, response.headers['content-type'])
assert response.status_code == 200

200 text/event-stream; charset=utf-8


## 이벤트 필드 관찰

토큰 이벤트는 재연결 위치에 쓸 `id`를 갖고 완료 이벤트는 별도 타입입니다.

In [3]:
import re
blocks = [b for b in re.split(r'\r?\n\r?\n', response.text) if b]
print(blocks[0])
print(blocks[-1])
assert blocks[0].splitlines()[0] == 'event: token'
assert 'id: sse-1:0' in blocks[0]
assert blocks[-1].splitlines()[0] == 'event: done'

event: token
data: {"token": "\uc9c8\ubb38\uc744 "}
id: sse-1:0
event: done
data: {"thread_id": "sse-1"}


## 반복 실험과 결과 해석

문장 길이만 바꾸면 토큰 이벤트 수는 달라지지만 완료 이벤트는 한 번입니다.

### 이 단계에서 구분할 범위

`token`과 `done`을 나누면 클라이언트는 문자열 조각과 실행 완료를 혼동하지 않습니다. 진행률, 출처, 도구 호출을 추가할 때도 별도 사건 이름을 사용하면 데이터 형식을 독립적으로 발전시킬 수 있습니다. 다만 SSE의 `id`는 업무 결과의 영구 식별자와 목적이 다르므로, 재연결 위치와 대화 식별자를 한 필드에 무리하게 합치지 않습니다.

In [4]:
for text in ['한', '두 단어']:
    res = client.post('/v1/chat/stream', headers=headers,
        json={'question': text, 'thread_id': 'count'})
    token_count = res.text.count('event: token')
    done_count = res.text.count('event: done')
    print(text, token_count, done_count)
    assert done_count == 1

한 3 1
두 단어 4 1
